# 차선 검출 - KITTI Sequence 09
본 과제에서는 Projection Matrix를 기반으로 3D 점의 투영 과정을 분석하고, 차량 궤적 시각화 및 Bayesian 기반 차선 검출을 수행하였다.  
또한 차선의 기하적 특성을 해석하고 딥러닝 모델과의 결과를 비교하였다.

#### 사용 데이터
- KITTI Odometry Dataset (Sequence 09)

#### 사용 패키지
- numpy
- opencv-python
- matplotlib
- Pillow

In [ ]:
# import
import numpy as np
import cv2
import matplotlib.pyplot as plt

# 문제 1. Projection Matrix 해석

Projection Matrix는 카메라의 내부 파라미터(K)와 외부 파라미터(R, t)를 결합하여,  
월드 좌표계의 3D 점을 카메라 좌표계를 거쳐 2D 이미지 평면으로 투영하는 변환 행렬이다.

전체 구조는 다음과 같다.

$$
P = K [R \mid t]
$$

---

### 1.1 Intrinsic Parameter (K)

$$
K =
\begin{bmatrix}
f_x & 0 & c_x \\
0 & f_y & c_y \\
0 & 0 & 1
\end{bmatrix}
$$

Intrinsic parameter는 카메라 내부 특성을 나타내며, 카메라 좌표계의 점을 이미지 좌표계로 변환하는 역할을 한다.

#### (1) \( f_x, f_y \) : 초점 거리
- 실제 카메라의 초점거리를 픽셀 단위로 표현한 값이다.
- 값이 클수록 이미지에서 물체가 더 크게 보인다.
- 이미지에서 확대 및 축소 정도를 결정한다.
- \( X_c / Z_c \), \( Y_c / Z_c \)에 곱해져 투영 스케일을 조정한다.

#### (2) \( c_x, c_y \) : 주점 (Principal Point)
- 이미지 중심 좌표를 의미한다.
- 카메라의 optical center를 나타낸다.
- 투영된 좌표를 이미지 중심 기준으로 이동시키는 역할을 한다.

Intrinsic parameter는 이미지에서의 위치와 크기를 결정하는 요소이다.

---

### 1.2 Extrinsic Parameter (\( R, t \))

$$
[R \mid t]
$$

Extrinsic parameter는 월드 좌표계의 3D 점을 카메라 좌표계로 변환하는 역할을 한다.

#### (1) \( R \) : 회전 행렬
- 카메라의 방향을 나타낸다.
- 카메라가 어느 방향을 바라보는지를 결정한다.

#### (2) \( t \) : 이동 벡터
- 카메라의 위치를 나타낸다.
- 카메라가 공간상 어디에 위치하는지를 정의한다.

Extrinsic parameter는 월드 좌표를 카메라 좌표로 변환하는 역할을 담당한다.

---

### 1.3 3D 점 → 이미지 좌표 변환 과정

#### (1) 카메라 좌표로 변환

$$
\begin{bmatrix}
X_c \\
Y_c \\
Z_c
\end{bmatrix}
=
R
\begin{bmatrix}
X \\
Y \\
Z
\end{bmatrix}
+ t
$$

월드 좌표계의 점을 카메라 기준 좌표계로 변환한다.

---

#### (2) 이미지 평면으로 투영

$$
\begin{bmatrix}
u' \\
v' \\
w'
\end{bmatrix}
=
P
\begin{bmatrix}
X \\
Y \\
Z \\
1
\end{bmatrix}
$$

---

#### (3) 최종 이미지 좌표 (Perspective Division)

$$
u = \frac{u'}{w'}, \quad v = \frac{v'}{w'}
$$

또는 다음과 같이 표현할 수 있다.

$$
u = \frac{f_x X_c}{Z_c} + c_x, \quad
v = \frac{f_y Y_c}{Z_c} + c_y
$$

---

### 1.4 핵심 해석

- \( X_c / Z_c \), \( Y_c / Z_c \)는 원근 효과를 나타내며, 물체가 멀어질수록 작게 보이는 현상을 설명한다.
- \( f_x, f_y \)는 이미지의 확대 및 축소를 결정한다.
- \( c_x, c_y \)는 이미지 중심 기준으로 좌표를 이동시킨다.

따라서 Projection Matrix는 카메라의 위치와 방향, 그리고 내부 파라미터를 반영하여  
3차원 공간의 점을 2차원 이미지 좌표로 변환하는 역할을 한다.

# 문제 2. Projection Matrix를 이용한 3D -> 2D 투영

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt


# =========================
# 1. Projection Matrix 읽기
# =========================
def read_projection_matrix(calib_path, camera="P1"):
    with open(calib_path, "r") as f:
        for line in f:
            if line.startswith(camera + ":"):
                values = list(map(float, line.split()[1:]))
                P = np.array(values).reshape(3, 4)
                return P


# =========================
# 2. 3D 점 생성
# =========================
def generate_3d_points():
    points = []

    for z in [10, 15, 20, 25, 30]:   # 거리
        for x in [-3, -2, -1, 0, 1, 2, 3]:  # 좌우
            y = 1.5   # 카메라 높이 근처
            points.append([x, y, z, 1])

    return np.array(points)


# =========================
# 3. Projection
# =========================
def project_points(P, points_3d):
    points_2d = []

    for X in points_3d:
        x_img = P @ X

        u = x_img[0] / x_img[2]
        v = x_img[1] / x_img[2]

        points_2d.append([u, v])

    return np.array(points_2d)


# =========================
# 4. 시각화
# =========================
def draw_points(image_path, points_2d):
    img = cv2.imread(image_path)

    for (u, v) in points_2d:
        u, v = int(u), int(v)

        if 0 <= u < img.shape[1] and 0 <= v < img.shape[0]:
            cv2.circle(img, (u, v), 5, (0, 0, 255), -1)

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(10,5))
    plt.imshow(img_rgb)
    plt.title("Projection of 3D points onto Image")
    plt.axis("off")
    plt.show()


# =========================
# 실행
# =========================
calib_path = "/Volumes/Data/.../calib.txt"
image_path = "/Volumes/Data/.../image_0/000000.png"

P = read_projection_matrix(calib_path)
points_3d = generate_3d_points()
points_2d = project_points(P, points_3d)

draw_points(image_path, points_2d)

### Projection 결과 해석

임의의 3차원 점들을 생성한 후 Projection Matrix를 이용하여 이미지 좌표로 투영하였다. 
그 결과, 동일한 간격으로 배치된 3D 점들이 이미지 상에서는 일정하지 않게 분포하는 것을 확인할 수 있었다.

먼저, 카메라로부터 가까운 점들은 이미지 하단에 크게 분포하고, 
멀리 있는 점들은 점점 위쪽으로 모이며 작게 표현된다. 
이는 투영 과정에서 \( X_c / Z_c \), \( Y_c / Z_c \) 형태로 계산되기 때문이며, 
거리 \( Z_c \)가 증가할수록 좌표값이 작아져 물체가 작게 보이는 원근 효과를 반영한다.

또한, 동일한 간격으로 배치된 3D 점들이 이미지에서는 점점 좁아지는 형태로 나타나는데, 
이는 실제 평행한 선들이 이미지 상에서 한 점으로 수렴하는 소실점(vanishing point) 현상과 일치한다.

이러한 결과는 카메라 투영의 주요 특성인 원근 효과와 소실점 형성이 잘 반영된 것으로 볼 수 있다. 
따라서 Projection Matrix를 이용한 변환이 실제 카메라의 이미지 형성 원리와 일치함을 확인할 수 있다.

# 문제 3. Pose를 이용한 차량 궤적 시각화

# 문제 4. Projection Matrix를 활용한 차선 해석

bayesian 분류를 통해 도로 영역을 추출한 후, 해당 영역에서 edge 검출과 Hough Transform을 적용하여 차선 후보를 추출하였다. 
검출된 차선은 이미지 좌표계 상에서는 직선의 형태로 나타난다.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


def extract_lane(mask):
    # edge 검출
    edges = cv2.Canny(mask, 50, 150)

    # Hough transform
    lines = cv2.HoughLinesP(
        edges,
        rho=1,
        theta=np.pi/180,
        threshold=50,
        minLineLength=50,
        maxLineGap=10
    )

    return lines, edges


def draw_lanes(image_path, lines):
    img = cv2.imread(image_path)

    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            cv2.line(img, (x1, y1), (x2, y2), (0,255,0), 3)

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(10,5))
    plt.imshow(img_rgb)
    plt.title("Lane Candidates")
    plt.axis("off")
    plt.show()


# 예시 실행
mask = cv2.imread("mask_000000.png", 0)
lines, edges = extract_lane(mask)
draw_lanes("000000.png", lines)

### 4.1 이미지 좌표에서 검출된 차선의 의미

이미지 상에서 검출된 차선은 2차원 좌표 \((u, v)\)로 표현되지만, 이는 실제 3차원 공간에서의 직선이 Projection Matrix에 의해 투영된 결과이다.  
즉, 이미지에서의 직선은 카메라 좌표계에서 특정 방향을 가지는 3D 직선의 투영이라고 볼 수 있다.

특히, 도로 위의 차선은 실제 공간에서는 서로 평행한 구조를 가지지만, 이미지에서는 서로 가까워지며 한 점으로 수렴하는 형태로 나타난다.

---

### 4.2 도로를 평면으로 가정할 때 차선의 기하적 특성

도로를 하나의 평면(ground plane)이라고 가정하면, 차선은 이 평면 위에 존재하는 직선이다.  
이러한 직선들은 3차원 공간에서 서로 평행한 관계를 가지며 일정한 간격으로 배치되어 있다.

그러나 카메라 투영 과정에서는 원근 효과에 의해 다음과 같은 특성이 나타난다.

- 가까운 차선은 크게 보이고 간격이 넓다
- 먼 차선은 작게 보이며 서로 가까워진다
- 평행한 직선들이 이미지 상에서는 한 점으로 수렴한다

이러한 현상은 실제 도로에서 차선이 멀어질수록 좁아 보이는 것과 동일한 원리이다.

---

### 4.3 Projection Matrix와 소실점 및 기울기의 관계

Projection Matrix는 3D 공간의 점을 이미지 평면으로 투영할 때 원근 효과를 반영한다.  
이 과정에서 평행한 직선들은 이미지 상에서 하나의 점으로 수렴하게 되며, 이를 소실점(vanishing point)이라고 한다.

차선의 경우, 실제 도로에서는 서로 평행하지만, 이미지에서는 동일한 방향으로 기울어진 직선으로 나타나며, 이 직선들은 상단의 특정 지점으로 수렴하게 된다.

이 소실점의 위치는 카메라의 방향(R)과 위치(t)에 의해 결정되며, Projection Matrix의 구성 요소에 의해 결정된다.  
즉, 카메라가 바라보는 방향이 달라지면 소실점의 위치도 함께 변하게 된다.

---

### 4.4 종합

따라서 이미지에서 검출된 차선은 단순한 2D 직선이 아니라,  
3차원 공간에서의 평행한 직선이 Projection Matrix에 의해 투영된 결과이며,  
원근 효과와 소실점 형성을 통해 실제 카메라 영상의 기하적 특성을 반영하고 있음을 확인할 수 있다.

# 문제 5. 실패 구간 분석

# 문제 6. 딥리서치 등을 활용해 차선 검출을 위한 딥러닝 모델을 찾아 하나를 제안하고, 적용한 결과를 비교하여 보이시오.